# Pak Dharti TV — Readership & Content Analysis

**Goal:** understand what readers of a 4-language news portal actually read, and whether the content pipeline spends AI budget where readers are.

**Data:** exported from the live site's admin panel (`/api/admin/export`) into `analysis/data/`:
- `pakdharti_articles.csv` — one row per article (category, type, origin, views per language, AI tokens)
- `pakdharti_daily.csv` — views per article per language per day

**Questions**
1. How much do we publish, and how is it split between automated and manually injected articles?
2. Which categories earn the most views per article?
3. Which languages do readers use, and is every translation worth its cost?
4. Do manual features outperform automated news?
5. What does one view cost in AI tokens, by category?
6. How does readership move over the week?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.spines.top": False, "axes.spines.right": False})

articles = pd.read_csv("data/pakdharti_articles.csv")
for col in ["created_at", "published_at"]:
    articles[col] = pd.to_datetime(articles[col], utc=True).dt.tz_convert(None)
daily = pd.read_csv("data/pakdharti_daily.csv", parse_dates=["day"])

print(articles.shape, daily.shape)
articles.head()

## Cleaning

In [ ]:
pub = articles[articles["status"] == "published"].copy()
pub["tokens"] = pub["input_tokens"] + pub["output_tokens"]
pub["pub_day"] = pub["published_at"].dt.floor("D")

# Data-quality checks worth reporting
print("Articles missing a published date:", pub["published_at"].isna().sum())
print("Missing translations:", {l: int((~pub[f"has_{l}"]).sum()) for l in ["ur", "ar", "de"]})
print("Duplicate slugs:", pub["slug"].duplicated().sum())

## 1. Publishing volume: automated vs manual

In [ ]:
volume = pub.groupby(["pub_day", "origin"]).size().unstack(fill_value=0)
volume.plot(kind="area", stacked=True, alpha=.8, title="Articles published per day")
plt.ylabel("articles"); plt.xlabel(""); plt.show()
volume.sum()

## 2. Views per article by category

In [ ]:
by_cat = (pub.groupby("category")
            .agg(articles=("slug", "count"), views=("views", "sum"), median_views=("views", "median"))
            .assign(views_per_article=lambda d: d.views / d.articles)
            .sort_values("views_per_article", ascending=False))
by_cat["views_per_article"].plot(kind="barh", title="Average views per article")
plt.gca().invert_yaxis(); plt.show()
by_cat.round(1)

## 3. Language share of readership

In [ ]:
lang_views = pub[["views_en", "views_ur", "views_ar", "views_de"]].sum()
lang_views.index = ["English", "Urdu", "Arabic", "German"]
share = (lang_views / lang_views.sum() * 100).round(1)
share.plot(kind="bar", title="Share of views by language (%)", rot=0); plt.show()
share

Each non-English language costs one extra AI call per article. If a language's share of views stays very low,
translating only top-performing articles into it (instead of all) would save budget.

## 4. Manual features vs automated news

In [ ]:
comparison = pub.groupby(["origin", "content_type"])["views"].describe()[["count", "mean", "50%", "75%"]]
pub.boxplot(column="views", by="origin", showfliers=False)
plt.suptitle(""); plt.title("Views per article by origin (outliers hidden)"); plt.show()
comparison.round(1)

Note: injected articles are often pinned to the breaking band, so they get more exposure.
A fair comparison would look at views *after* the pin expires, or compare only unpinned articles.

## 5. AI cost per view

In [ ]:
# Set these to your provider's current prices (USD per million tokens)
PRICE_IN, PRICE_OUT = 3.0, 15.0

pub["cost_usd"] = pub["input_tokens"] / 1e6 * PRICE_IN + pub["output_tokens"] / 1e6 * PRICE_OUT
cost = (pub.groupby("category")
          .agg(total_cost=("cost_usd", "sum"), views=("views", "sum"))
          .assign(cost_per_1k_views=lambda d: d.total_cost / d.views.replace(0, pd.NA) * 1000)
          .sort_values("cost_per_1k_views"))
print(f"Total AI spend: ${pub['cost_usd'].sum():.2f}  |  avg per article: ${pub['cost_usd'].mean():.3f}")
cost.round(3)

## 6. Readership over the week

In [ ]:
d = daily.copy()
d["weekday"] = d["day"].dt.day_name()
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekly = d.groupby(["weekday", "lang"])["views"].sum().unstack(fill_value=0).reindex(order)
weekly.plot(kind="bar", stacked=True, rot=30, title="Views by weekday and language"); plt.show()

## Findings

*Fill in after running on real data — 3 to 5 sentences, each tied to a number above. For example:*

- Category X earns N× the views per article of category Y, but receives only Z% of output.
- Language L accounts for N% of views; translating only the top 20% of articles into it would cut AI spend by about $M/month.

## Recommendations

*What would you change in the content pipeline based on these numbers?*